In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

/data/users/goodarzilab/shervin/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
class GCNModel(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=80, output_dim=1, dropout=0.3):
        super(GCNModel, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim, normalize=False)
        self.conv2 = GCNConv(hidden_dim, hidden_dim, normalize=False)
        self.conv3 = GCNConv(hidden_dim, 64, normalize=False)
        self.conv4 = GCNConv(64, output_dim, normalize=False)
        self.dropout = dropout
        
        self.layer1_activations = None
        self.layer2_activations = None
        self.layer3_activations = None
        self.layer4_activations = None

    def forward(self, data, store_activations=False):
        x, edge_index = data.x, data.edge_index
        edge_weight = getattr(data, 'edge_attr', None)

        h1 = F.relu(self.conv1(x, edge_index, edge_weight=edge_weight))
        if store_activations: self.layer1_activations = h1.detach()
        h1 = F.dropout(h1, p=self.dropout, training=self.training)

        h2 = F.relu(self.conv2(h1, edge_index, edge_weight=edge_weight))
        if store_activations: self.layer2_activations = h2.detach()
        h2 = F.dropout(h2, p=self.dropout, training=self.training)

        h3 = F.relu(self.conv3(h2, edge_index, edge_weight=edge_weight))
        if store_activations: self.layer3_activations = h3.detach()
        h3 = F.dropout(h3, p=self.dropout, training=self.training)

        h4 = self.conv4(h3, edge_index, edge_weight=edge_weight)
        if store_activations: self.layer4_activations = h4.detach()

        return h4.squeeze(-1)

    def get_activations(self):
        return self.layer1_activations, self.layer2_activations, self.layer3_activations, self.layer4_activations

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCNModel(input_dim=2, hidden_dim=80, output_dim=1)
checkpoint = torch.load("gnn_model.pt", map_location=device)
model.load_state_dict(checkpoint)
model

GCNModel(
  (conv1): GCNConv(2, 80)
  (conv2): GCNConv(80, 80)
  (conv3): GCNConv(80, 64)
  (conv4): GCNConv(64, 1)
)

In [9]:
activations_path = "../outputs/activations/layer3_new/train/graph_0.pt"
activation = torch.load(activations_path, map_location=device)
activation.shape

torch.Size([10, 64])